### compare_avg_regrid_methods.py
Compare your current "avg" ice regridding (bilinear -> 0.1deg -> box-mean
coarsen to 1deg) against a true conservative (area-weighted) regrid straight
from the native POP tripolar grid to 1deg.

Assumes you're running this in the same session/notebook where you already
built: nat_ice_hr, mask_ice_hr, grid_ice_hr, dst_1deg, bbox, bbox_regrid,
regridder_ice_to_0p1deg, coarsen_files, ice_vars, etc. It reuses those
directly rather than rebuilding them.

### What it does:
1. Builds an approximate conservative regridder for the native ice grid,
   using POP's own U-point locations as the cell corners (exact model
   geometry, not an averaged approximation) -- falls back to a
   neighbor-averaging corner estimate if ULONG/ULAT aren't available.
2. Regrids one real field (default: "aice", first available timestep of the
   first ensemble member) two ways: your current avg method, and the new
   conservative method.
3. Plots both fields side by side plus their difference.
4. Plots the native grid geometry (via the corners) near your domain so you
   can visually check whether the tripole singularity (nominally in the
   Yukon Territory) intrudes into your masked region -- that's the zone
   where the two methods would disagree most.

In [7]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
import pop_tools
import glob

In [8]:
def collect_files(dirs, vars, start_year):
    out = []
    for d in dirs:
        member_files = {}
        for v in vars:
            c = comps[v]
            pattern = f"{d}/{c}/proc/tseries/month_1/*.{v}.*.nc"
            files = sorted(glob.glob(pattern))

            # Keep only files from 1920/2006 on
            filtered = []
            for f in files:
                year = int(f.split('.')[-2][:4])
                if year >= start_year:
                    filtered.append(f)

            member_files[v] = filtered

        out.append(member_files)
    return out


# ---------- Directories on glade ----------

low_res_dirs = sorted(
    glob.glob('/glade/campaign/collections/gdex/data/d651030/BHIST/*')  # /BRCP85
)
high_res_dirs = sorted(
    glob.glob('/glade/campaign/collections/gdex/data/d651007/b.e13.*')  # /d651009
)

# ---------- Variables ----------

low_vars = ['hi', 'aice', 'U', 'V']
comps = {
    'hi': 'ice',
    'aice': 'ice',
    'U': 'atm',
    'V': 'atm',
}

target_var = ['hi']

# ---------- Collect files ----------

low_res_files = collect_files(low_res_dirs, low_vars, start_year=1920)  # 2006
high_res_files = collect_files(high_res_dirs, target_var, start_year=1920)  # 2006
coarsen_files = collect_files(high_res_dirs, low_vars, start_year=1920)  # 2006

print('Low-res  | # ens:', len(low_res_files), '| # vars:', len(low_res_files[0]))
print('High-res | # ens:', len(high_res_files), '| # vars:', len(high_res_files[0]))
print('Coarsen  | # ens:', len(coarsen_files), '| # vars:', len(coarsen_files[0]))

Low-res  | # ens: 10 | # vars: 4
High-res | # ens: 10 | # vars: 1
Coarsen  | # ens: 10 | # vars: 4


In [ ]:
# =====================================================================
# Get native grids for Kivalina region
# =====================================================================

# ---------- Region select ----------

# Larger region for regridding
bbox_regrid = {"lon_min": -200, "lon_max": -130, "lat_min": 55, "lat_max": 85}
lon_min_regrid = bbox_regrid["lon_min"] % 360
lon_max_regrid = bbox_regrid["lon_max"] % 360

# Actual ML domain
bbox = {"lon_min": -190, "lon_max": -140, "lat_min": 60, "lat_max": 80}
lon_min = bbox["lon_min"] % 360
lon_max = bbox["lon_max"] % 360

print("Region select done.")

# ---------- Native grids ----------

atm_dir = "/glade/p/cesmdata/cseg/inputdata/share/scripgrids/"
# nat_atm_lr = xr.open_dataset(atm_dir + "ne30np4_091226_pentagons.nc")
nat_atm_hr = xr.open_dataset(atm_dir + "ne120np4_pentagons_100310.nc")

atm_lat = nat_atm_hr.grid_center_lat.values
atm_lon = nat_atm_hr.grid_center_lon.values % 360

mask_atm_hr = (
    (atm_lat >= bbox_regrid["lat_min"])
    & (atm_lat <= bbox_regrid["lat_max"])
    & (atm_lon >= lon_min_regrid)
    & (atm_lon <= lon_max_regrid)
)

grid_atm_hr = xr.Dataset({
    "lat": ("ncol", nat_atm_hr.grid_center_lat.values[mask_atm_hr]),
    "lon": ("ncol", nat_atm_hr.grid_center_lon.values[mask_atm_hr] % 360),
})

# nat_ice_lr = pop_tools.get_grid("POP_gx1v7")
nat_ice_hr = pop_tools.get_grid("POP_tx0.1v2")

ice_lon = nat_ice_hr.TLONG % 360

mask_ice_hr = np.any(
    ((nat_ice_hr.TLAT >= bbox_regrid["lat_min"])
     & (nat_ice_hr.TLAT <= bbox_regrid["lat_max"])
     & (ice_lon >= lon_min_regrid)
     & (ice_lon <= lon_max_regrid)),
    axis=1,
)

grid_ice_hr = xr.Dataset({
    "lat": (["nlat", "nlon"], nat_ice_hr.TLAT.isel(nlat=mask_ice_hr).values),
    "lon": (["nlat", "nlon"], ice_lon.isel(nlat=mask_ice_hr).values),
})

print("Native grids to regrid prepared.")

# ---------- Destination grids ----------

dst_1deg = xr.Dataset({
    "lat": ("lat", np.arange(bbox_regrid["lat_min"], bbox_regrid["lat_max"] + 1, 1.0)),
    "lon": ("lon", np.arange(lon_min_regrid, lon_max_regrid + 1, 1.0)),
})

dst_0p1deg = xr.Dataset({
    "lat": ("lat", np.arange(bbox_regrid["lat_min"], bbox_regrid["lat_max"] + 0.1, 0.1)),
    "lon": ("lon", np.arange(lon_min_regrid, lon_max_regrid + 0.1, 0.1)),
})

print("Destination grids set up.")

# ---------- Build regridders ----------

WEIGHTED_GRIDS_DIR = "/glade/work/skygale/_projects/SeaIceDownscaling/weighted_grids"

# Ice interpolated (coarsened)
regridder_ice_to_1deg_interp = xe.Regridder(
    grid_ice_hr,
    dst_1deg,
    method="bilinear",
    periodic=True,
    filename=f"{WEIGHTED_GRIDS_DIR}/ice_hr_to_1deg_interp.nc",
    reuse_weights=True,
)

print(" >> Built regridder_ice_to_1deg_interp.")

# Ice target & grid cell average (coarsened) -- source grid pre-masked to the
# Kivalina bbox_regrid, same weight matrix as 3d_data_process.ipynb's
# regridder_ice_to_0p1deg, hence the shared filename
regridder_ice_to_0p1deg = xe.Regridder(
    grid_ice_hr,
    dst_0p1deg,
    method="bilinear",
    periodic=True,
    filename=f"{WEIGHTED_GRIDS_DIR}/ice_hr_to_0p1deg_kivalina_masked.nc",
    reuse_weights=True,
)

print(" >> Built regridder_ice_to_0p1deg.")

# Atm interpolated (coarsened)
regridder_atm_to_1deg_interp = xe.Regridder(
    grid_atm_hr,
    dst_1deg,
    method="nearest_s2d",
    locstream_in=True,
    periodic=False,
    filename=f"{WEIGHTED_GRIDS_DIR}/atm_hr_to_1deg_interp.nc",
    reuse_weights=True,
)

print(" >> Built regridder_atm_to_1deg_interp.")

print("Regridder weight files done/reused.")

In [ ]:
# =====================================================================
# 1. Build corners for the native ice grid (POP U-points = NE corner of
#    each T-cell). This gives the true model cell geometry rather than an
#    averaged approximation.
# =====================================================================

has_upoints = hasattr(nat_ice_hr, "ULONG") and hasattr(nat_ice_hr, "ULAT")
print("Native U-point corners available:", has_upoints)

tlon_full = (nat_ice_hr.TLONG.values % 360)
tlat_full = nat_ice_hr.TLAT.values
ny_full, nx_full = tlon_full.shape

rows = np.where(mask_ice_hr)[0]
r0, r1 = rows.min(), rows.max()
contiguous = np.array_equal(rows, np.arange(r0, r1 + 1))
if not contiguous:
    print("WARNING: masked rows are not contiguous in the native nlat index -- "
          "the vertex-grid corner construction below assumes contiguity and "
          "may be wrong. Treat conservative results with extra caution.")

if r0 < 1 or r1 + 1 > ny_full - 1:
    print("WARNING: masked region touches the native grid's j-edge -- corner "
          "construction near the true grid boundary/pole fold is not handled "
          "here. Results near that edge may be unreliable.")

if has_upoints:
    ulon_full = (nat_ice_hr.ULONG.values % 360)
    ulat_full = nat_ice_hr.ULAT.values
    # Pad one ghost column in i (periodic longitude) so vertex grid has nx+1 cols.
    ulon_i = np.pad(ulon_full, ((0, 0), (1, 0)), mode="wrap")
    ulat_i = np.pad(ulat_full, ((0, 0), (1, 0)), mode="wrap")
    # Vertex grid for the masked rows: need native rows r0-1 .. r1 (n_rows+1 total).
    lon_b_sub = ulon_i[r0 - 1:r1 + 1, :]
    lat_b_sub = ulat_i[r0 - 1:r1 + 1, :]
else:
    # Fallback: approximate corners by averaging neighboring T-cell centers.
    print("Falling back to averaged-neighbor corner approximation (less "
          "accurate near strong grid distortion, e.g. near the tripole).")

    def _extend(a):
        ny, nx = a.shape
        out = np.empty((ny + 2, nx + 2))
        out[1:-1, 1:-1] = a
        out[0, 1:-1] = 2 * a[0, :] - a[1, :]
        out[-1, 1:-1] = 2 * a[-1, :] - a[-2, :]
        out[1:-1, 0] = 2 * a[:, 0] - a[:, 1]
        out[1:-1, -1] = 2 * a[:, -1] - a[:, -2]
        out[0, 0] = 2 * a[0, 0] - a[1, 1]
        out[0, -1] = 2 * a[0, -1] - a[1, -2]
        out[-1, 0] = 2 * a[-1, 0] - a[-2, 1]
        out[-1, -1] = 2 * a[-1, -1] - a[-2, -2]
        return out

    lon_e, lat_e = _extend(tlon_full), _extend(tlat_full)
    lon_b_full = 0.25 * (lon_e[:-1, :-1] + lon_e[:-1, 1:] + lon_e[1:, :-1] + lon_e[1:, 1:])
    lat_b_full = 0.25 * (lat_e[:-1, :-1] + lat_e[:-1, 1:] + lat_e[1:, :-1] + lat_e[1:, 1:])
    lon_b_sub = lon_b_full[r0:r1 + 2, :]
    lat_b_sub = lat_b_full[r0:r1 + 2, :]

grid_ice_hr_conserv = xr.Dataset({
    "lat": (["nlat", "nlon"], tlat_full[r0:r1 + 1, :]),
    "lon": (["nlat", "nlon"], tlon_full[r0:r1 + 1, :]),
    "lat_b": (["nlat_b", "nlon_b"], lat_b_sub),
    "lon_b": (["nlat_b", "nlon_b"], lon_b_sub),
})

# =====================================================================
# 2. Destination 1deg grid corners (exact -- trivial for a regular grid)
# =====================================================================

dst_lat_c = dst_1deg.lat.values
dst_lon_c = dst_1deg.lon.values
dst_lat_b = np.concatenate([[dst_lat_c[0] - 0.5], dst_lat_c + 0.5])
dst_lon_b = np.concatenate([[dst_lon_c[0] - 0.5], dst_lon_c + 0.5])
dst_1deg_b = xr.Dataset({
    "lat": ("lat", dst_lat_c),
    "lon": ("lon", dst_lon_c),
    "lat_b": ("lat_b", dst_lat_b),
    "lon_b": ("lon_b", dst_lon_b),
})

# =====================================================================
# 3. Build the conservative regridder
# =====================================================================

print("Building conservative regridder (this can take a bit longer than "
      "bilinear/nearest since it computes cell-overlap areas)...")
regridder_ice_conservative = xe.Regridder(
    grid_ice_hr_conserv,
    dst_1deg_b,
    method="conservative",
    periodic=True,
    filename="/glade/work/skygale/_projects/SeaIceDownscaling/weighted_grids/ice_hr_to_1deg_conservative.nc",
    reuse_weights=False,  # set True on reruns once you trust the weight file
)
print(" >> Built conservative regridder.")

# =====================================================================
# 4. Load one real field/timestep and regrid both ways
# =====================================================================

test_var = "aice"
test_file = coarsen_files[0][test_var][0]
print("Test file:", test_file)

ds = xr.open_dataset(test_file)
da = ds[test_var].rename({"nj": "nlat", "ni": "nlon"})
da = da.isel(nlat=mask_ice_hr)
da_t0 = da.isel(time=0)

# Method A: current avg method (bilinear -> 0.1deg -> box mean -> 1deg)
da_hr = regridder_ice_to_0p1deg(da_t0)
da_avg_A = da_hr.coarsen(lat=10, lon=10, boundary="trim", coord_func="min").mean()
da_avg_A = da_avg_A.assign_coords(
    lat=np.round(da_avg_A.lat.values, 6),
    lon=np.round(da_avg_A.lon.values, 6),
)
da_avg_A = da_avg_A.sel(
    lat=slice(bbox["lat_min"], bbox["lat_max"]),
    lon=slice(lon_min, lon_max),
)

# Method B: true conservative regrid, native -> 1deg directly
da_avg_B = regridder_ice_conservative(da_t0)
da_avg_B = da_avg_B.sel(
    lat=slice(bbox["lat_min"], bbox["lat_max"]),
    lon=slice(lon_min, lon_max),
)

diff = da_avg_A - da_avg_B

print("Method A (bilinear+box-mean) stats:", float(da_avg_A.min()), float(da_avg_A.max()))
print("Method B (conservative) stats:     ", float(da_avg_B.min()), float(da_avg_B.max()))
print("Diff (A - B) stats:                ", float(diff.min()), float(diff.max()),
      "| mean abs diff:", float(np.abs(diff).mean()))

# =====================================================================
# 5. Plots
# =====================================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)

im0 = da_avg_A.plot(ax=axes[0], add_colorbar=False)
axes[0].set_title(f"{test_var}: bilinear + box-mean (current)")
fig.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = da_avg_B.plot(ax=axes[1], add_colorbar=False)
axes[1].set_title(f"{test_var}: conservative (area-weighted)")
fig.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = diff.plot(ax=axes[2], add_colorbar=False, cmap="RdBu_r",
                 vmin=-np.abs(diff).max(), vmax=np.abs(diff).max())
axes[2].set_title("Difference (A - B)")
fig.colorbar(im2, ax=axes[2], shrink=0.8)

plt.savefig("avg_method_comparison.png", dpi=150)
print("Saved avg_method_comparison.png")

# =====================================================================
# 6. Sanity check: does the tripole singularity intrude into your domain?
#    Plot the native grid mesh geometry (via the corners just computed) so
#    you can see any anomalously tiny/distorted cells.
# =====================================================================

fig2, ax2 = plt.subplots(figsize=(7, 6))
sc = ax2.scatter(
    grid_ice_hr_conserv.lon.values.ravel(),
    grid_ice_hr_conserv.lat.values.ravel(),
    c=grid_ice_hr_conserv.lat.values.ravel(),
    s=1, cmap="viridis",
)
ax2.set_xlim(lon_min_regrid, lon_max_regrid)
ax2.set_ylim(bbox_regrid["lat_min"], bbox_regrid["lat_max"])
ax2.set_title("Native ice grid nodes in your region (color = TLAT)\n"
              "Look for irregular spacing/clustering -- a sign the tripole\n"
              "singularity (nominally in Yukon Territory) is nearby.")
fig.colorbar(sc, ax=ax2, shrink=0.8, label="TLAT (deg)")
plt.savefig("native_grid_check.png", dpi=150)
print("Saved native_grid_check.png")

print("\nDone. Max |TLAT| in masked region:", float(np.abs(tlat_full[r0:r1 + 1, :]).max()))